# 🕌 Bulaq 1280 AH ByT5 Arabic OCR Corrector — v2 (Sentence-Level)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefbouhaik/bulaq-ocr-transformer/blob/main/Train_Bulaq_Transformer_Colab.ipynb)

Fine-tunes **`google/byt5-small`** on **69,640 sentence-level** Bulaq 1280 AH OCR correction pairs.

### ⚡ v2 Fixes (from failed v1)
| Problem | v1 (Failed) | v2 (Fixed) |
|:---|:---|:---|
| Dataset granularity | 60% single-word pairs | Full sentence pairs (15-120 chars) |
| Punctuation-only pairs | 32% of dataset | < 3% of dataset |
| Learning rate | 1e-3 (gradient explosion) | 3e-4 (stable convergence) |
| Dataset size | 28,468 word fragments | 69,640 sentence pairs |
| Training signal | Add comma / fix one letter | Multi-error sentence healing |

## 1. Check Hardware Acceleration
*Works on TPU v5e, A100 GPU, L4 GPU, or CPU (slower)*

In [ ]:
# Check for GPU
!nvidia-smi 2>/dev/null || echo 'No GPU detected'

# Check for TPU
try:
    import torch_xla.core.xla_model as xm
    print('✅ TPU Active:', xm.xla_device())
except Exception:
    print('No TPU detected (will use GPU or CPU)')

## 2. Install Packages & Clone Repo

In [ ]:
!pip install -q transformers[torch] datasets accelerate
# Always get latest version of the repo
!rm -rf bulaq-ocr-transformer
!git clone https://github.com/youssefbouhaik/bulaq-ocr-transformer.git
%cd bulaq-ocr-transformer
!ls -lh data/

## 3. Load & Inspect Sentence-Level Dataset

In [ ]:
import json, gzip
from datasets import Dataset

data_path = "data/bulaq_sentence_pairs.jsonl.gz"
pairs = []
with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            src = row.get("corrupt_ocr", "").strip()
            tgt = row.get("ground_truth", "").strip()
            if src and tgt and src != tgt:
                pairs.append({"input_text": src, "target_text": tgt})

print(f"✅ Loaded {len(pairs):,} sentence-level correction pairs!")

# Show sample pairs
for i in range(5):
    print(f"\n{i+1}. ❌ OCR:   {pairs[i]["input_text"][:80]}...")
    print(f"   ✅ Truth: {pairs[i]["target_text"][:80]}...")

# Split 90% train, 10% validation
raw_ds = Dataset.from_list(pairs).train_test_split(test_size=0.1, seed=42)
train_ds = raw_ds["train"]
val_ds = raw_ds["test"]
print(f"\nTrain: {len(train_ds):,} | Val: {len(val_ds):,}")

## 4. Initialize ByT5 Model & Tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/byt5-small'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Detect device
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
except Exception:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
print(f'✅ Model loaded on {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Tokenize Dataset

In [ ]:
max_src_len = 256
max_tgt_len = 256

def preprocess(batch):
    inputs = tokenizer(
        batch['input_text'],
        max_length=max_src_len,
        padding='max_length',
        truncation=True
    )
    targets = tokenizer(
        text_target=batch['target_text'],
        max_length=max_tgt_len,
        padding='max_length',
        truncation=True
    )
    labels = [
        [(t if t != tokenizer.pad_token_id else -100) for t in target]
        for target in targets['input_ids']
    ]
    inputs['labels'] = labels
    return inputs

print('Tokenizing train set...')
tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=['input_text', 'target_text'])
print('Tokenizing validation set...')
tokenized_val = val_ds.map(preprocess, batched=True, remove_columns=['input_text', 'target_text'])
print(f'✅ Tokenized: {len(tokenized_train):,} train, {len(tokenized_val):,} val')

## 6. Training Configuration & Execution
*With an A100/TPU and BF16, 3 epochs takes ~15-20 minutes on 69K pairs.*

In [ ]:
import inspect
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# ⚡ Explicitly register torch.xla namespace so Hugging Face recognizes TPU
is_tpu = False
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    is_tpu = True
    print(f"✅ TPU Hardware Initialized: {device}")
except Exception:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        model.gradient_checkpointing_enable()
        print("Running on CUDA GPU")
    else:
        print("Running on CPU")

# Optimal TPU vs GPU hyperparameter routing
batch_size = 16 if is_tpu else 4
grad_accum = 2 if is_tpu else 8

training_args = Seq2SeqTrainingArguments(
    output_dir="./bulaq_byt5_checkpoints",
    optim="adamw_torch",
    learning_rate=3e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=3,
    warmup_steps=200,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=25,
    bf16=is_tpu,                         # ⚡ Native BF16 for TPU TensorCores
    fp16=False if is_tpu else torch.cuda.is_available(),
    gradient_checkpointing=False if is_tpu else True, # Static graph on TPU
    predict_with_generate=False,
    report_to="none",
)

if hasattr(training_args, "eval_strategy"):
    training_args.eval_strategy = "epoch"
elif hasattr(training_args, "evaluation_strategy"):
    training_args.evaluation_strategy = "epoch"

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
}
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__)
if "processing_class" in trainer_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("🚀 Starting ByT5 fine-tuning on 69,640 Bulaq sentence pairs...")
trainer.train()


## 7. Test The Trained Model

In [ ]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
eval_model = trainer.model.to("cpu")
eval_model.eval()

# Visual correspondence sibling pools (physical lithographic proximity)
VISUAL_SIBLINGS = {
    "ا": ["ا", "ل"], "ل": ["ل", "ا"],
    "د": ["د", "ذ"], "ذ": ["ذ", "د", "غ"],
    "ع": ["ع", "غ"], "غ": ["غ", "ع", "ذ"],
    "ف": ["ف", "ق"], "ق": ["ق", "ف"],
    "ب": ["ب", "ت", "ث", "ن", "ي"],
    "ت": ["ت", "ث", "ب", "ن", "ي"],
    "ث": ["ث", "ت", "ب", "ن", "ي"],
    "ن": ["ن", "ي", "ت", "ب", "ث"],
    "ي": ["ي", "ى", "ن", "ب", "ت"],
    "ه": ["ه", "ة"], "ة": ["ة", "ه"],
}

def heal_with_guided_correspondence(raw_ocr):
    """
    Heals raw OCR by combining pre-trained language model context
    with visual lithographic correspondence steering.
    """
    inputs = tokenizer(raw_ocr, return_tensors="pt").to("cpu")
    target_len = len(raw_ocr.encode("utf-8")) + 12
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=target_len,
            num_beams=4,
            repetition_penalty=1.4,
            no_repeat_ngram_size=12,
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_cases = [
    "بلذنى ايها الماك السعيد ان ملكا من ملوك ساسان",
    "فلما سمع الملك شهريار هده الحكايه من شهرزاد",
    "والله لقدضاقت بى الارض لاجل غيبتك",
    "فنظر الى المرأه وهى قاعده على كرسى من الدهب",
    "قال لها يا سيدتى انا رجل غريب وقد تعبث من السفر",
    "ثم ان اخي دخل على الخليفه وقبل الارض بين يديه",
]

print("=== 🕌 BULAQ 1280 AH BYT5 HEALING (GUIDED INFERENCE) ===
")
for raw in test_cases:
    healed = heal_with_guided_correspondence(raw)
    print(f"❌ Corrupt OCR : {raw}")
    print(f"✅ Healed Text : {healed}")
    print("-" * 60)


## 8. Save Trained Model

In [ ]:
# Save locally
output_dir = './bulaq_byt5_ocr_corrector'
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'✅ Model saved to {output_dir}')

# Optional: Save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p '/content/drive/MyDrive/bulaq_byt5_ocr_corrector'
    !cp -r ./bulaq_byt5_ocr_corrector/* '/content/drive/MyDrive/bulaq_byt5_ocr_corrector/'
    print('✅ Model backed up to Google Drive!')
except Exception as e:
    print(f'Google Drive save skipped: {e}')

## 9. 🔍 Edition Collation & Word-by-Word Alignment Tool
*Compare your physical refined OCR transcription against our Bulaq dataset ground truth to detect editorial variations vs OCR typos.*

In [ ]:
import difflib
from IPython.display import HTML, display

def compare_ocr_against_dataset(refined_ocr_text, dataset_reference_text):
    """
    Performs fine-grained token-level collation between your physical refined OCR
    and the dataset ground-truth text, computing the exact word match percentage
    and highlighting editorial differences.
    """
    words_refined = refined_ocr_text.strip().split()
    words_ref = dataset_reference_text.strip().split()
    
    matcher = difflib.SequenceMatcher(None, words_ref, words_refined)
    similarity = matcher.ratio() * 100
    
    print(f"════════════════════════════════════════════════════════════════")
    print(f"📊 WORD-BY-WORD ALIGNMENT REPORT")
    print(f"   Refined OCR Word Count : {len(words_refined):,} words")
    print(f"   Dataset Word Count     : {len(words_ref):,} words")
    print(f"   Overall Match Rate     : {similarity:.2f}%")
    print(f"════════════════════════════════════════════════════════════════
")
    
    html_diff = ["<div style='font-family: Arial, sans-serif; font-size: 16px; direction: rtl; line-height: 2.2;'>"]
    diff_count = 0
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            html_diff.append(f"<span style='color: #2e7d32;'> {' '.join(words_ref[i1:i2])} </span>")
        elif tag == "replace":
            diff_count += 1
            ref_part = ' '.join(words_ref[i1:i2])
            ocr_part = ' '.join(words_refined[j1:j2])
            html_diff.append(f"<span style='background-color: #ffebee; color: #c62828; text-decoration: line-through; padding: 2px 4px; border-radius: 3px;'> {ref_part} </span>")
            html_diff.append(f"<span style='background-color: #e8f5e9; color: #2e7d32; font-weight: bold; padding: 2px 4px; border-radius: 3px;'> {ocr_part} </span>")
        elif tag == "delete":
            diff_count += 1
            html_diff.append(f"<span style='background-color: #fff3e0; color: #ef6c00; text-decoration: line-through;'> {' '.join(words_ref[i1:i2])} </span>")
        elif tag == "insert":
            diff_count += 1
            html_diff.append(f"<span style='background-color: #e3f2fd; color: #1565c0; font-weight: bold;'> {' '.join(words_refined[j1:j2])} </span>")
            
    html_diff.append("</div>")
    print(f"Identified {diff_count} divergence points (Red = Dataset, Green = Your Refined OCR):
")
    display(HTML("".join(html_diff)))

# Example Test Comparison (Paste your own refined excerpt here!)
sample_refined_ocr = "فلما سمع الملك شهريار هذه الحكاية من شهرزاد قال لها يا شهرزاد لقد سرني هذا الحديث"
sample_dataset_ref = "فلما سمع الملك شهريار هده الحكايه من شهرزاد قال لها يا شهرزاد لقد اعجبني هذا الحديث"

compare_ocr_against_dataset(sample_refined_ocr, sample_dataset_ref)
